In [ ]:
!apt-get update
!apt install chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium

# 1. Crawl images 10 loại rau củ quả

**Đã chạy hoàn thành ở version 3**

In [ ]:
from selenium.webdriver.chrome.options import Options
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
import time
import pandas as pd
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import requests


from io import BytesIO
from PIL import Image

from concurrent.futures import ThreadPoolExecutor

from time import sleep
import os

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

DATA_ROOT = "/content/drive/MyDrive/vegetable_dataset"
RAW_ROOT = os.path.join(DATA_ROOT, "raw")
PROCESSED_ROOT = os.path.join(DATA_ROOT, "processed")
SPLITS_ROOT = os.path.join(DATA_ROOT, "splits")
FILTERED_IMAGE_ROOT = os.path.join(DATA_ROOT, "filtered-image")
os.makedirs(RAW_ROOT, exist_ok=True)
os.makedirs(PROCESSED_ROOT, exist_ok=True)
os.makedirs(SPLITS_ROOT, exist_ok=True)
os.makedirs(FILTERED_IMAGE_ROOT, exist_ok=True)

In [ ]:
def get_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument("--disable-gpu")
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.5672.63 Safari/537.36")
    options.add_argument("--window-size=1051x798")
    options.add_argument("--incognito")
    driver = webdriver.Chrome(options=options)
    driver.set_window_size(1051, 798)
    return driver

In [ ]:
# link thiết lập sẵn chế độ tìm kiếm
crawl_link = "https://www.freepik.com/search?ai=excluded&format=search&last_filter=people&last_value=exclude&people=exclude&sort=relevance&type=photo"


classifications = ["tomato", "potato", "carrot", "banana", "broccoli", "eggplant",  "corn", "pineapple", "orange", "asparagus"]
keywords =       ["fresh tomato fruit", "raw potato", "fresh carrot", "banana fruit", "green fresh broccoli",
                   "fresh eggplant", "corn cob", "pineapple fruit", "orange", "fresh asparagus"]

userAgent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0"


In [ ]:
def crawl_for_keyword(classification, keyword, crawl_link, userAgent):
    try:
        # fake agent sang firefox tránh spam khi request

        print("Bắt đầu tải ảnh cho từ khóa:", keyword)
        driver = get_driver()
        driver.get(crawl_link)

        folder = os.path.join(RAW_ROOT, classification)
        if not os.path.exists(folder):
            os.makedirs(folder)
            print(f"Đã tạo thư mục: {folder}")

        # Tìm kiếm từ khóa
        search_input = driver.find_element(By.CSS_SELECTOR, 'input[data-cy="search-photo"]')
        search_input.send_keys(keyword)
        search_input.send_keys(Keys.ENTER)

        time.sleep(2)

        page = 1
        count = 1
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(4)  # Chờ một chút để trang tải xong và tránh bị chặn

            searchedImagesInPage = driver.find_elements(By.CSS_SELECTOR, 'img.\\$rounded.\\$object-cover')
            for image in searchedImagesInPage:
                image_link = image.get_attribute('src')
                if image_link:
                    try:
                        headers = {
                            "User-Agent": userAgent
                        }
                        response = requests.get(image_link, headers=headers, timeout=20)
                        if response.status_code == 200:
                            # Mở ảnh từ bytes
                            img = Image.open(BytesIO(response.content)).convert('RGB')  # convert về RGB để lưu JPG chuẩn

                            # Tạo tên file duy nhất với định dạng .jpg
                            file_name = f"{classification}{count}.jpg"
                            file_path = os.path.join(folder, file_name)

                            # Lưu ảnh dưới dạng JPG
                            img.save(file_path, 'JPEG')
                            # print(f"Đã lưu: {file_path}")
                            count += 1
                        else:
                            print(f"Tải lỗi (status {response.status_code}): {image_link}")
                    except Exception as e:
                        print(f"Lỗi xảy ra {image_link}: {e}")

            # Nhấn nút sang trang cho đến khi hết (trang cho phép max 100 trang)
            try:
                nextPageButton = driver.find_element(By.CSS_SELECTOR, 'a[data-cy="pagination-next"]')
                driver.get(nextPageButton.get_attribute('href'))
            except NoSuchElementException:
                print("Không còn trang nào nữa.")
                break
            time.sleep(1)
            if page % 10 == 0:
                print(f"{classification} Đã tới trang: {page}/100")
                time.sleep(10) #tránh  bị chặn ip
            page += 1
        print("Hoàn tất việc tải ảnh cho từ khóa:", keyword)
        print("Tổng số ảnh đã tải:", count)
    finally:
        driver.quit()

In [ ]:
def run_crawlers(keywords, classifications, crawl_link, userAgent):
    with ThreadPoolExecutor(max_workers=10) as executor:
        for i in range(len(keywords)):
            classification = classifications[i]
            keyword = keywords[i]
            executor.submit(crawl_for_keyword, classification, keyword, crawl_link, userAgent)

In [ ]:
run_crawlers(keywords, classifications, crawl_link, userAgent)

# 2. Xử lí ảnh trùng lặp

**Đã chạy ở version 5**

In [ ]:
import hashlib
import os
import shutil
from tqdm import tqdm

In [ ]:
def md5(fname):
    hash_md5 = hashlib.md5()
    with open(fname, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

input_root = RAW_ROOT
output_root = PROCESSED_ROOT

# Lấy danh sách folder con
folders = [f for f in os.listdir(input_root) if os.path.isdir(os.path.join(input_root, f))]

total_kept = 0
total_skipped = 0

for folder in folders:
    src_folder = os.path.join(input_root, folder)
    dest_folder = os.path.join(output_root, folder)
    os.makedirs(dest_folder, exist_ok=True)

    print(f"Xử lý thư mục: {folder}")
    files = [f for f in os.listdir(src_folder) if f.lower().endswith(('.jpg'))]

    # Mỗi thư mục có một tập hợp hash riêng
    folder_hashes = set()

    for filename in tqdm(files, desc=f"Copy ảnh trong {folder}"):
        src_path = os.path.join(src_folder, filename)
        dest_path = os.path.join(dest_folder, filename)
        try:
            filehash = md5(src_path)
            if filehash not in folder_hashes:
                folder_hashes.add(filehash)
                shutil.copy2(src_path, dest_path)
                total_kept += 1
            else:
                total_skipped += 1
        except Exception as e:
            print(f"Lỗi xử lý file {src_path}: {e}")

print(f"Tổng ảnh giữ lại: {total_kept}")
print(f"Tổng ảnh trùng bỏ qua: {total_skipped}")

In [ ]:
folders = [f for f in os.listdir(output_root) if os.path.isdir(os.path.join(output_root, f))]

for folder in folders:
    folder_path = os.path.join(output_root, folder)

    # Lấy các file .jpg trong thư mục hiện tại
    jpg_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.jpg')]
    count = len(jpg_files)

    print(f" {folder}: {count} ảnh .jpg")

# 3. Lọc thủ công

*tool hỗ trợ [Github](https://github.com/Chinh-de/ImageDatasetFilterTool)*

In [ ]:
import json
import random
from math import floor
import shutil
import os

seed = 42
random.seed(seed)

In [ ]:
folder_path = FILTERED_IMAGE_ROOT
data_dict = {}

for file_name in os.listdir(folder_path):
    if file_name.endswith(".json"):
        file_path = os.path.join(folder_path, file_name)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            key = os.path.splitext(file_name)[0]
            data_dict[key] = data.get("verified", [])

print(data_dict["tomato"][0])

In [ ]:
source_root = PROCESSED_ROOT
dest_root = SPLITS_ROOT
random.seed(42)

splits = ["train", "val", "test"]
ratios = [0.7, 0.15, 0.15]

# Tạo thư mục đầu ra
for split in splits:
    for class_name in data_dict:
        os.makedirs(os.path.join(dest_root, split, class_name), exist_ok=True)

for class_name, file_list_full in data_dict.items():
    file_list = file_list_full[:1000]
    src_folder = os.path.join(source_root, class_name)
    random.shuffle(file_list)  # Xáo trộn ảnh

    total = len(file_list)
    n_train = floor(ratios[0] * total)
    n_val = floor(ratios[1] * total)
    n_test = total - n_train - n_val

    splits_count = {
        "train": file_list[:n_train],
        "val": file_list[n_train:n_train+n_val],
        "test": file_list[n_train+n_val:]
    }

    print(f"\nClass '{class_name}' split:")
    print(f"  Train ({len(splits_count['train'])})")
    print(f"  Val   ({len(splits_count['val'])})")
    print(f"  Test  ({len(splits_count['test'])})")

    for split, files in splits_count.items():
        for fname in files:
            src_path = os.path.join(src_folder, fname)
            dst_path = os.path.join(dest_root, split, class_name, fname)
            if os.path.exists(src_path):
                shutil.copy(src_path, dst_path)